In [0]:
# 1. importing BeautifulSoup library

try:
    from bs4 import BeautifulSoup
    print("BeautifulSoup library has been imported successfully")
except:
    print("BeautifulSoup library installation or import issue occured")

In [0]:
# 2. Master ingestion function

import requests, os
from datetime import datetime, timedelta
from bs4 import BeautifulSoup


# Azure storage config

storage_account_name = "REMOVED_FOR_SECURITY"
storage_account_key = "REMOVED_FOR_SECURITY"

spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)

# Auto create folders if missing

spark.conf.set("fs.azure.createRemoteFileSystemDuringInitialization", "true")


# Reusable ingestion function

def ingest_dataset(dataset_name, page_url, file_pattern):
    print(f"starting ingestion for dataset {dataset_name.upper()}")
    
    # define dynamic paths
    base_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/{dataset_name}"
    landing_zone = f"{base_path}/landing/"
    archive_zone = f"{base_path}/archive/"
    
    # A. CSV file Link finder
    print(f"finding download link on page : {page_url}")
    headers = {'User_Agent':'Mozilla/5.0'}
    response = requests.get(page_url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.content,"html.parser")

    fresh_url = None

    for link in soup.find_all('a', href='True'):
        if 'download' in link['href'] and file_pattern in link['href']:
            fresh_url = f"https://www.dubaipulse.gov.ae{link['href']}" if link['href'].startswith('/') else link['href']
            break
    if not fresh_url : raise Exception(f"link not found for {file_pattern}")


    # B. Archive current file
    
    try:
        dbutils.fs.ls(f"{landing_zone}current.csv")
        today = datetime.now().strftime("%Y-%m-%d")
        dbutils.fs.mv(f"{landing_zone}current.csv",f"{archive_zone}{today}.csv")
        print("Archived old files")
    except : pass

    # C. Download and upload

    local_path = f"/tmp/{dataset_name}.csv"
    print("Downloading and uploading to ADLS")

    with requests.get(fresh_url, headers=headers, stream=True) as r:
        r.raise_for_status()
        with open({local_path, 'wb'}) as f:
            for chunk in r.iter_content(chunk_size=10*1024*1024):
                f.write(chunk)
    
    dbutils.fs.cp(f"file:{local_path}", f"{landing_zone}current.csv")
    os.remove(local_path)
    print(f"FINISHED : Datset {dataset_name} is ready.")




print("Master Function Loaded")



In [0]:
# 3. Execute Ingestion

# Ingest Units
ingest_dataset(
    dataset_name="units2",
    page_url="https://www.dubaipulse.gov.ae/data/dld-registration/dld_units-open",
    file_pattern="units.csv"
)

ingest_dataset(
    dataset_name="products2",
    page_url="https://www.dubaipulse.gov.ae/data/dld-registration/dld_products-open",
    file_pattern="products.csv"
)